Squad 2 | Streaming em Tempo Real

 | **Objetivo** | Análise exploratória das tabelas do Squad 2 |

| **Tabelas** | ecommerce_categorias, ecommerce_itens_pedido, ecommerce_produtos |

| **Fonte** | ADLS Gen2 → snapshot mais recente |

| **Depende de** | feat_squad2_99_helpers |

In [0]:
%run ../utils/feat_squad2_99_helpers

In [0]:
# Suprimir logs verbosos do Azure SDK
import logging
logging.getLogger("azure").setLevel(logging.WARNING)

inicio = log_inicio("feat_squad2_03_analise_exploratoria")


In [0]:
from pyspark.sql.functions import col, sum as spark_sum, when, min as spark_min, max as spark_max

try:
    snapshot_id = get_snapshot_mais_recente()
    log.info(f"Snapshot selecionado: {snapshot_id}\n")

    dataframes = {}
    for tabela in TABELAS_SQUAD2:
        df                 = ler_parquet(snapshot_id, tabela)
        dataframes[tabela] = df
        log.info(
            f"OK {tabela} → "
            f"{df.count()} linhas | "
            f"{len(df.columns)} colunas"
        )

except Exception as e:
    log.error(f"Erro ao carregar tabelas: {str(e)}")
    raise

In [0]:
for tabela, df in dataframes.items():
    print("=" * 60)
    print(f"  TABELA: {tabela.upper()}")
    print("=" * 60)

    print("\n SCHEMA:")
    df.printSchema()

    print("\n AMOSTRA:")
    display(df)

    print(f"\n TOTAL DE REGISTROS: {df.count()}\n")

In [0]:
for tabela, df in dataframes.items():
    print("=" * 60)
    print(f"  NULOS: {tabela.upper()}")
    print("=" * 60)

    df_nulos = df.select([
        spark_sum(
            when(col(c).isNull(), 1).otherwise(0)
        ).alias(c)
        for c in df.columns
    ])
    display(df_nulos)
    print("\n")

In [0]:
for tabela, df in dataframes.items():
    print("=" * 60)
    print(f"  ESTATÍSTICAS: {tabela.upper()}")
    print("=" * 60)
    display(df.describe())
    print("\n")

In [0]:
for tabela, df in dataframes.items():
    total = df.count()
    print("=" * 60)
    print(f"  CHAVES CANDIDATAS: {tabela.upper()}")
    print("=" * 60)

    for coluna in df.columns:
        distinct = df.select(coluna).distinct().count()
        pct      = round((distinct / total) * 100, 1)
        flag     = " CANDIDATA A PK" if distinct == total else ""
        print(f"  {coluna}: {distinct} distintos / {total} total ({pct}%) {flag}")
    print("\n")

In [0]:
# Validar integridade referencial: produtos → categorias
try:
    df_produtos    = dataframes["ecommerce_produtos"]
    df_categorias  = dataframes["ecommerce_categorias"]

    # Produtos com categoria inexistente
    ids_categorias = df_categorias.select("id_categoria")
    df_orfaos      = df_produtos.join(
        ids_categorias,
        df_produtos["id_categoria"] == ids_categorias["id_categoria"],
        "left_anti"
    )

    total_orfaos = df_orfaos.count()

    if total_orfaos == 0:
        log.info(" Integridade referencial OK — todos os produtos têm categoria válida")
    else:
        log.warning(f" {total_orfaos} produto(s) com id_categoria sem correspondência")
        display(df_orfaos)

except Exception as e:
    log.error(f"Erro na validação referencial: {str(e)}")

In [0]:
print("=" * 60)
print("   RESUMO EXECUTIVO — SQUAD 2")
print("=" * 60)

for tabela, df in dataframes.items():
    total    = df.count()
    colunas  = len(df.columns)
    nulos    = df.select([
        spark_sum(when(col(c).isNull(), 1).otherwise(0))
        for c in df.columns
    ]).collect()[0]
    total_nulos = sum(nulos)

    print(f"\n  Pacote {tabela.upper()}")
    print(f"     Registros : {total}")
    print(f"     Colunas   : {colunas}")
    print(f"     Nulos     : {total_nulos}")
    print(f"     Qualidade : {' OK' if total_nulos == 0 else ' Verificar'}")

print("\n" + "=" * 60)

log_fim("feat_squad2_03_analise_exploratoria", inicio)